    # Lab 06 · Autoencoder: Unsupervised Learning on MNIST
    **AI Lab · University of Technology Bahrain** · ⏱ ~15 min · ⚡ GPU recommended

    **You will learn**
    - Learning without labels: compress → reconstruct
- A 2-D latent space you can plot (compare with PCA)
- Generating new digits by decoding points in the latent space

    > **First time on Kaggle?** Make sure your account is phone-verified, then in the right-hand panel open
    > **Settings → Accelerator** (choose *GPU T4 x2* when the lab needs it) and **Settings → Internet → On**.
    > Run cells top to bottom with **Shift + Enter**.

In [ ]:
# ⚙️ Setup: run me first
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

ON_KAGGLE = os.path.exists("/kaggle/working")
OUT_DIR = "/kaggle/working" if ON_KAGGLE else "."
gpus = tf.config.list_physical_devices("GPU")
print("Running on Kaggle:", ON_KAGGLE)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", [g.name for g in gpus] if gpus else "none (CPU). Turn on Settings > Accelerator > GPU if this lab needs it")
keras.utils.set_random_seed(42)

In [ ]:
# 📦 Dataset loader: downloads with Keras, falls back to datasets attached via "+ Add Input"
def load_dataset(name):
    """name: 'mnist', 'fashion_mnist' or 'cifar10' -> (x_train, y_train), (x_test, y_test)"""
    try:
        return getattr(keras.datasets, name).load_data()
    except Exception as err:
        if name == "mnist":  # Kaggle-attached copy of mnist.npz, if any
            hits = glob.glob("/kaggle/input/**/mnist.npz", recursive=True)
            if hits:
                d = np.load(hits[0])
                return (d["x_train"], d["y_train"]), (d["x_test"], d["y_test"])
        raise RuntimeError(
            "Could not download the dataset. On Kaggle: right panel > Settings > Internet > On "
            "(needs a phone-verified account), then run this cell again.") from err

In [ ]:
(x_train, y_train), (x_test, y_test) = load_dataset("mnist")
x_train = x_train.astype("float32").reshape(-1, 784) / 255.0
x_test = x_test.astype("float32").reshape(-1, 784) / 255.0

## 1 · Encoder → 2 numbers → Decoder
The labels are **never** used for training, only later to colour the plot.

In [ ]:
LATENT = 2
encoder = keras.Sequential([keras.Input((784,)), layers.Dense(256, activation="relu"),
                            layers.Dense(64, activation="relu"), layers.Dense(LATENT)], name="encoder")
decoder = keras.Sequential([keras.Input((LATENT,)), layers.Dense(64, activation="relu"),
                            layers.Dense(256, activation="relu"), layers.Dense(784, activation="sigmoid")], name="decoder")
autoencoder = keras.Sequential([encoder, decoder])
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
autoencoder.fit(x_train, x_train, epochs=15, batch_size=256, validation_data=(x_test, x_test), verbose=2)

In [ ]:
rec = autoencoder.predict(x_test[:10], verbose=0)
fig, axes = plt.subplots(2, 10, figsize=(12, 2.6))
for i in range(10):
    axes[0, i].imshow(x_test[i].reshape(28, 28), cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(rec[i].reshape(28, 28), cmap="gray"); axes[1, i].axis("off")
axes[0, 0].set_title("original", loc="left"); axes[1, 0].set_title("reconstructed (from 2 numbers!)", loc="left")
plt.show()

## 2 · The latent space vs PCA

In [ ]:
from sklearn.decomposition import PCA
z = encoder.predict(x_test, verbose=0)
p = PCA(2).fit_transform(x_test)
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
for a, pts, title in [(ax[0], p, "PCA (linear)"), (ax[1], z, "Autoencoder (non-linear)")]:
    sc = a.scatter(pts[:, 0], pts[:, 1], c=y_test, cmap="tab10", s=2)
    a.set_title(title)
fig.colorbar(sc, ax=ax, ticks=range(10)); plt.show()

## 3 · Generate digits by walking through the latent space

In [ ]:
n = 15
xs = np.linspace(np.percentile(z[:, 0], 3), np.percentile(z[:, 0], 97), n)
ys = np.linspace(np.percentile(z[:, 1], 3), np.percentile(z[:, 1], 97), n)
grid = np.array([[x, y] for y in ys[::-1] for x in xs])
imgs = decoder.predict(grid, verbose=0).reshape(n, n, 28, 28)
plt.figure(figsize=(8, 8))
plt.imshow(imgs.transpose(0, 2, 1, 3).reshape(n * 28, n * 28), cmap="gray"); plt.axis("off")
plt.title("Decoded grid of latent points"); plt.show()

## 🧪 Try it
1. Set `LATENT = 16`. Reconstructions improve, but how could you still visualise the latent space? (Hint: PCA or t-SNE on `z`.)
2. Run K-means with 10 clusters on `z`. How well do the clusters match the true digits?
3. Train on Fashion-MNIST instead. Which classes overlap in the latent plot?

### 💾 Save your work
Click **Save Version** (top right) → *Quick Save* to keep your edits, or *Save & Run All* to store the outputs too.
Files written to `/kaggle/working` appear under **Output** and can be downloaded.